# Dependencies

In [ ]:
%%capture

# Core training libraries
!pip install -q \
    transformers==4.44.2 \
    datasets==2.20.0 \
    tokenizers==0.19.1 \
    accelerate==0.34.2 \
    peft==0.12.0 \
    trl==0.9.6 \
    bitsandbytes==0.43.1 \
    evaluate==0.4.2

# Utilities
!pip install -q \
    numpy \
    pandas \
    scikit-learn \
    rich \
    pyyaml \
    python-dotenv \
    tqdm

# Evaluation (requires pydantic v2)
!pip install -q --upgrade pydantic
!pip install -q google-genai rouge-score

print("✅ Installation complete!")
print("✅ All dependencies compatible (pydantic v2 + google-genai)")

# ENV Update

In [ ]:
# Verify it's loaded
from dotenv import load_dotenv
load_dotenv()
# Show only key names for security
try:
    with open('.env', 'r') as f:
        print("🔑 Keys in .env file:")
        print("="*60)
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                key = line.split('=')[0]
                value_preview = line.split('=')[1][:10] + "..." if '=' in line else ""
                print(f"  {key} = {value_preview}")
        print("="*60)
except FileNotFoundError:
    print("❌ .env file not found")

# GPU check

In [ ]:
import sys
import torch

print("="*60)
print("ENVIRONMENT CHECK")
print("="*60)
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"Device capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ WARNING: CUDA not available. Training will be VERY slow on CPU.")

print("="*60)

# Seeds

In [ ]:
import os
import random
import numpy as np
import torch

SEED = 42

# Set environment variable for Python hash seed
os.environ['PYTHONHASHSEED'] = str(SEED)

# Set seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    # Note: These settings may impact performance
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"✅ Seeds set to {SEED} for reproducibility")
print("⚠️ Note: Full determinism on GPU is not guaranteed due to non-deterministic operations")

# Login HF

In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')
!hf auth login --token $HF_TOKEN

print("ℹ️ Hugging Face login skipped. Uncomment login() to push models to Hub.")

# Config

In [ ]:
import torch
from pprint import pprint

# Auto-detect compute dtype (BF16 requires compute capability >= 8.0)
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

CONFIG = {
    # Model
    "base_model": "meta-llama/Llama-3.1-8B",
    # Alternative for tighter VRAM: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    # For GGUF export, prefer: "meta-llama/Llama-3.2-3B-Instruct" or Mistral models

    # Dataset
    "dataset_name": "janadaj/operation-ledger-mind-qa",
    "dataset_split": "train",
    "dataset_subsample": 500,  # Colab-safe: 500 | Local: 1500
    "train_val_split": 0.9,  # 90% train, 10% validation

    # Tokenization
    "max_length": 512,  # Colab: 512 | Local: 1024

    # Training
    "num_train_epochs": 10,
    "max_steps": 250,  # Colab: 250 | Local: 600
    "per_device_train_batch_size": 1,  # Colab: 1 | Local: 2
    "gradient_accumulation_steps": 64,  # Colab: 64 | Local: 32
    "learning_rate": 2e-5,
    "warmup_ratio": 0.03,
    "logging_steps": 10,
    "save_steps": 200,
    "eval_steps": 100,
    "save_total_limit": 2,

    # LoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],

    # Quantization
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": compute_dtype,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": True,

    # Output
    "output_dir": "outputs/adapter",
    "push_to_hub": False,

    # Generation
    "max_new_tokens": 128,
    "temperature": 0.0,  # Deterministic
    "do_sample": True,

    # HF credentials
    'hf_username': 'janadaj',
    'hub_model_name': 'operation-ledger-mind-qa',
}

# Effective batch size
effective_batch_size = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]

print("="*60)
print("CONFIGURATION (COLAB FREE TIER)")
print("="*60)
pprint(CONFIG)
print("="*60)
print(f"Compute dtype: {compute_dtype}")
print(f"Using BF16: {use_bf16}")
print(f"Effective batch size: {effective_batch_size}")
print("="*60)

# Get Data from HF

In [ ]:
from datasets import load_dataset, Dataset
import json
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# Read token for private Hugging Face dataset
HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("hf_token")


def load_hf_dataset(dataset_name, subsample, split="train", seed=42):
    """Load dataset with robust field mapping and fallback"""

    try:
        # Try loading from Hugging Face with authentication
        print(f"📥 Loading dataset: {dataset_name} (split='{split}')...")
        dataset = load_dataset(dataset_name, split=split, token=HF_TOKEN)
        dataset = dataset.shuffle(seed=seed).select(range(min(subsample, len(dataset))))
        print(f"✅ Loaded {len(dataset)} examples from Hugging Face")
        return dataset
        
    except Exception as e:
        print(f"⚠️ Failed to load from Hugging Face: {e}")
        print("🔄 Creating synthetic fallback dataset...")

        templates = [
            {
                "qa_id": "chunk_0001_q01",
                "chunk_id": 1,
                "question": "What is Uber's core mission statement as described in the 2024 annual report?",
                "answer": "We reimagine the way the world moves for the better.",
                "context": "Uber’s Mission: We reimagine the way the world moves for the better. We are Uber. The go-getters. The kind of people who are relentless about our mission to help people go anywhere and get anything and earn their way. Movement is what we power. It’s our lifeblood. It runs through our veins.",
                "metadata": {
                    "source": "data/pdfs/AEE Mini Project 01 2024 Annual Report.pdf",
                    "page": 1,
                    "splitter": "recursive"
                }
            },
            {
                "qa_id": "chunk_0019_q07",
                "chunk_id": 19,
                "question": "What benefits do Uber One members receive according to the report?",
                "answer": "Uber One members receive discounts, special pricing, priority service, and perks across Mobility and Delivery.",
                "context": "Membership: With our platform, we are making it even easier for our consumers to unlock value. Uber One is our single cross-platform membership program where members receive discounts, special pricing, and priority service across all consumer offerings.",
                "metadata": {
                    "source": "data/pdfs/AEE Mini Project 01 2024 Annual Report.pdf",
                    "page": 19,
                    "splitter": "recursive"
                }
            },
            {
                "qa_id": "chunk_0029_q08",
                "chunk_id": 29,
                "question": "What role does the Chief People Officer have in relation to the CEO?",
                "answer": "The Chief People Officer reports directly to the Chief Executive Officer (CEO).",
                "context": "Executive Leadership: Our senior management team oversees strategy and operations. The Chief People Officer directs human resources, workplace culture, and workforce engagement, reporting directly to the CEO.",
                "metadata": {
                    "source": "data/pdfs/AEE Mini Project 01 2024 Annual Report.pdf",
                    "page": 29,
                    "splitter": "recursive"
                }
            },
            {
                "qa_id": "chunk_0035_q04",
                "chunk_id": 35,
                "question": "In what year was the company originally incorporated as Ubercab, Inc.?",
                "answer": "The company was originally incorporated as Ubercab, Inc. in 2010.",
                "context": "Corporate History: The company was originally incorporated in Delaware in 2010 under the name Ubercab, Inc., and subsequently changed its legal name to Uber Technologies, Inc. in February 2011.",
                "metadata": {
                    "source": "data/pdfs/AEE Mini Project 01 2024 Annual Report.pdf",
                    "page": 35,
                    "splitter": "recursive"
                }
            },
            {
                "qa_id": "chunk_0048_q01",
                "chunk_id": 48,
                "question": "What was the total Gross Bookings reported for the fiscal year 2024?",
                "answer": "Gross Bookings grew 20% year-over-year to $162.6 billion in 2024.",
                "context": "Financial Performance: In 2024, Gross Bookings grew 20% year-over-year to $162.6 billion. Mobility Gross Bookings reached $82.7 billion (up 24%), and Delivery Gross Bookings reached $74.5 billion (up 17%).",
                "metadata": {
                    "source": "data/pdfs/AEE Mini Project 01 2024 Annual Report.pdf",
                    "page": 48,
                    "splitter": "recursive"
                }
            },
            {
                "qa_id": "chunk_0052_q09",
                "chunk_id": 52,
                "question": "What competitive advantages might Delivery competitors hold over the company?",
                "answer": "Greater resources and access to larger driver, consumer, and merchant networks in specific markets.",
                "context": "Competition: Our Delivery offering competes with iFood, Delivery Hero, Just Eat Takeaway, and Amazon. Competitors may possess greater brand recognition, larger regional merchant networks, or access to larger capital resources.",
                "metadata": {
                    "source": "data/pdfs/AEE Mini Project 01 2024 Annual Report.pdf",
                    "page": 52,
                    "splitter": "recursive"
                }
            },
            {
                "qa_id": "chunk_0069_q02",
                "chunk_id": 69,
                "question": "What was Uber's GAAP operating income for the year 2024?",
                "answer": "Operating income was $2.8 billion, demonstrating sustained GAAP profitability.",
                "context": "Operating Results: For the year ended December 31, 2024, operating income was $2.8 billion, compared to $1.1 billion in 2023, representing sustained operating profitability across core operations.",
                "metadata": {
                    "source": "data/pdfs/AEE Mini Project 01 2024 Annual Report.pdf",
                    "page": 69,
                    "splitter": "recursive"
                }
            },
            {
                "qa_id": "chunk_0169_q04",
                "chunk_id": 169,
                "question": "What type of vehicle licenses are currently frozen for ridesharing in certain cities?",
                "answer": "For-hire vehicle licenses.",
                "context": "Regulatory Environment: Certain municipal regulatory authorities have imposed caps or freezes on for-hire vehicle licenses for ridesharing services like those enabled via our platform.",
                "metadata": {
                    "source": "data/pdfs/AEE Mini Project 01 2024 Annual Report.pdf",
                    "page": 169,
                    "splitter": "recursive"
                }
            }
        ]

        synthetic_data = []
        for i in range(subsample):
            item = templates[i % len(templates)].copy()
            item["qa_id"] = f"fallback_{item['chunk_id']:04d}_q{i:03d}"
            synthetic_data.append(item)

        # In-memory Dataset creation (reliable on Windows, Linux & Colab)
        dataset = Dataset.from_list(synthetic_data)
        print(f"✅ Created synthetic dataset with {len(dataset)} examples")
        return dataset


# Load train set
train = load_hf_dataset(
    dataset_name=CONFIG["dataset_name"],
    subsample=CONFIG["dataset_subsample"],
    split=CONFIG.get("dataset_split", "train"),
    seed=SEED
)

# preview data

In [ ]:
import pandas as pd
# Convert first 50 samples to dataframe
df_preview = pd.DataFrame(train[:50])
# Display with formatting
pd.set_option('display.max_colwidth', 100)  # Limit column width for readability
print(f"📊 Displaying first {len(df_preview)} samples out of {len(train)} total examples\n")

display(df_preview)

### Token Length & Truncation Diagnostics

In [ ]:
from transformers import AutoTokenizer
import numpy as np

print(f"Loading tokenizer: {CONFIG['base_model']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['base_model'],trust_remote_code=True)

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

sample_size = min(500, len(train))
sample_dataset = train.select(range(sample_size))

token_lengths=[]
for example in sample_dataset:
    # Concatenate question answer and context
    text = f"{example['question']} {example['answer']} {example['context']}"
    tokens = tokenizer(text, add_special_tokens=True)
    token_lengths.append(len(tokens["qa_id"]))

token_lengths= np.array(token_lengths)

print("="*60)
print("TOKEN LENGTH DIAGNOSTICS")
print("="*60)
print(f"Sample size: {sample_size}")
print(f"Average token length: {token_lengths.mean():.1f}")
print(f"Median token length: {np.median(token_lengths):.1f}")
print(f"Min token length: {token_lengths.min()}")
print(f"Max token length: {token_lengths.max()}")
print(f"95th percentile: {np.percentile(token_lengths, 95):.1f}")
print(f"99th percentile: {np.percentile(token_lengths, 99):.1f}")
print()
truncated = (token_lengths > CONFIG["max_length"]).sum()
truncation_rate = truncated / len(token_lengths) * 100
print(f"Truncation at max_length={CONFIG['max_length']}: {truncated}/{len(token_lengths)} ({truncation_rate:.1f}%)")
print("="*60)

### Build SFT Prompts

In [ ]:
SYSTEM_PROMPT = (
    "You are an authoritative financial analysis assistant. "
    "Answer the user's question strictly based on the provided financial context. "
    "If the answer cannot be determined directly from the context, respond with: "
    "'Based on the provided document, this information is not available.'"
)
def chatml_format(user_text, system_text=SYSTEM_PROMPT, assistant_text=None):
    """Format in ChatML format"""
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]
    if assistant_text:
        messages.append({"role": "assistant", "content": assistant_text})
    # Render ChatML tokens
    formatted = ""
    for msg in messages:
        formatted += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
    return formatted


def build_sft_prompt(row):
    """Build SFT prompt from dataset row using (question, context, answer)."""
    # 1. Question
    question = row.get("question") 
    # 2. Context (replaces 'input')
    context = row.get("context") 
    if context:
        user_text = f"Context:\n\"\"\"{context}\"\"\"\n\nQuestion:\n{question}"
    else:
        user_text = question
    # 3. Answer (replaces 'output')
    answer = row.get("answer")
    prompt = chatml_format(
        user_text=user_text,
        system_text=SYSTEM_PROMPT,
        assistant_text=answer
    )
    return {"text": prompt}

train_dataset = train.map(build_sft_prompt)
print("✅ Prompts built for SFT")
print("\n📝 Sample formatted prompt:")
print(train_dataset[0]["text"][:500] + "...")

## Baseline inference (Before FT)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Quantization configuration (4-bit QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=CONFIG["load_in_4bit"],
    bnb_4bit_compute_dtype=CONFIG["bnb_4bit_compute_dtype"],
    bnb_4bit_quant_type=CONFIG["bnb_4bit_quant_type"],
    bnb_4bit_use_double_quant=CONFIG["bnb_4bit_use_double_quant"],
)

print(f"📥 Loading base model: {CONFIG['base_model']} in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
base_model.eval()

print("✅ Base model loaded in 4-bit successfully!")
if torch.cuda.is_available():
    print(f"  VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

In [ ]:
import time
import torch

# Financial QA Test Prompts for Baseline Evaluation (Before Fine-Tuning)
test_prompts = [
    {
        "title": "Financial Metric Extraction (Gross Bookings)",
        "prompt": (
            "Context:\n"
            "\"\"\"Financial Performance: In 2024, Gross Bookings grew 20% year-over-year to $162.6 billion. "
            "Mobility Gross Bookings reached $82.7 billion (up 24%), and Delivery Gross Bookings reached $74.5 billion (up 17%).\"\"\"\n\n"
            "Question:\nWhat was the total Gross Bookings reported for the fiscal year 2024?"
        )
    },
    {
        "title": "Operational Membership Benefits (Uber One)",
        "prompt": (
            "Context:\n"
            "\"\"\"Membership: With our platform, we are making it even easier for our consumers to unlock value. "
            "Uber One is our single cross-platform membership program where members receive discounts, special pricing, "
            "and priority service across all consumer offerings.\"\"\"\n\n"
            "Question:\nWhat benefits do Uber One members receive according to the report?"
        )
    },
    {
        "title": "Unanswerable / Fallback Query",
        "prompt": (
            "Context:\n"
            "\"\"\"Uber’s Mission: We reimagine the way the world moves for the better. We are Uber. The go-getters. "
            "The kind of people who are relentless about our mission to help people go anywhere and get anything.\"\"\"\n\n"
            "Question:\nWhat was the company's average employee turnover rate in Europe for 2024?"
        )
    },
]

print("=" * 60)
print("BASELINE OUTPUTS (PRE-FINETUNE)")
print("=" * 60)

if torch.cuda.is_available():
    vram_before = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM before generation: {vram_before:.2f} GB\n")

# Generation configuration
use_sampling = CONFIG.get("do_sample", False) and (CONFIG.get("temperature", 0.0) > 0)

for i, test in enumerate(test_prompts, 1):
    # 1. Format as ChatML with assistant generation prompt
    formatted_prompt = chatml_format(test["prompt"])
    if not formatted_prompt.endswith("<|im_start|>assistant\n"):
        formatted_prompt += "<|im_start|>assistant\n"

    # 2. Tokenize input
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(base_model.device)
    input_len = inputs["input_ids"].shape[1]

    # 3. Generate completion
    start_time = time.time()
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=CONFIG.get("max_new_tokens", 128),
            do_sample=use_sampling,
            temperature=CONFIG["temperature"] if use_sampling else None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    elapsed = time.time() - start_time

    # 4. Extract ONLY the newly generated tokens
    new_tokens = outputs[0][input_len:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # Clean up trailing ChatML tokens if present
    response = response.replace("<|im_end|>", "").strip()

    tokens_generated = len(new_tokens)
    tokens_per_sec = tokens_generated / elapsed if elapsed > 0 else 0

    print(f"\n[{i}] {test['title']}")
    print("-" * 60)
    print(f"Prompt preview: {test['prompt'][:100]}...")
    print(f"\nResponse:\n{response}")
    print(f"\nLatency: {elapsed:.2f}s | Tokens: {tokens_generated} | Speed: {tokens_per_sec:.1f} tok/s")
    print("-" * 60)

if torch.cuda.is_available():
    vram_after = torch.cuda.memory_allocated() / 1024**3
    print(f"\nVRAM after generation:  {vram_after:.2f} GB")
    print(f"VRAM delta:             {vram_after - vram_before:.2f} GB")

print("=" * 60)

## 8. Model + LoRA Setup and Training

Prepare the model for k-bit training, apply LoRA, and train with SFTTrainer.

In [ ]:
# ----------------------------------------------------------------------------
# Supervised Fine-Tuning (SFT) with QLoRA
# ----------------------------------------------------------------------------
import os
import builtins
os.environ["ACCELERATE_USE_DEEPSPEED"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

# 0. Fix upstream bug in trl: inject missing BaseTunerLayer into sft_trainer namespace
try:
    from peft.tuners.tuners_utils import BaseTunerLayer
except ImportError:
    class BaseTunerLayer:
        pass

builtins.BaseTunerLayer = BaseTunerLayer
try:
    import trl.trainer.sft_trainer
    trl.trainer.sft_trainer.BaseTunerLayer = BaseTunerLayer
except Exception:
    pass

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

# Ensure trl is available if session was restarted
try:
    import trl
    from trl import SFTTrainer, SFTConfig
except ModuleNotFoundError:
    import sys, subprocess
    print("📦 'trl' not found in active environment. Installing trl...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "trl==0.9.6", "peft==0.12.0"])
    import trl
    from trl import SFTTrainer, SFTConfig

from transformers.utils import is_bitsandbytes_available
import bitsandbytes as bnb
import torch

# Ensure key variables are defined
seed = globals().get("SEED", 42)
use_bf16 = globals().get(
    "use_bf16",
    torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
)
effective_batch_size = (
    CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]
)

# 1. Prepare Train & Validation splits
print("📊 Preparing train/validation datasets...")
if "val_dataset" in globals() and val_dataset is not None:
    train_split = train_dataset
    val_split = val_dataset
    print("  Using existing 'val_dataset' from notebook environment.")
else:
    # Attempt to load the 'test' split uploaded to Hugging Face
    try:
        print(f"📥 Loading test split from Hugging Face for evaluation: {CONFIG['dataset_name']}...")
        test_raw = load_hf_dataset(
            dataset_name=CONFIG["dataset_name"],
            subsample=min(100, CONFIG.get("dataset_subsample", 500)),
            split="test",
            seed=seed
        )
        val_split = test_raw.map(build_sft_prompt)
        train_split = train_dataset
        print(f"✅ Loaded {len(val_split)} validation examples from HF 'test' split!")
    except Exception as e:
        print(f"ℹ️ Could not load test split ({e}). Creating validation split from train_dataset...")
        val_ratio = 1.0 - CONFIG.get("train_val_split", 0.9)
        split_data = train_dataset.train_test_split(test_size=val_ratio, seed=seed)
        train_split = split_data["train"]
        val_split = split_data["test"]

print(f"  Training samples:   {len(train_split):,}")
print(f"  Validation samples: {len(val_split):,}")

# 2. Prepare base model for k-bit training with gradient checkpointing
print("
🔧 Preparing model for k-bit training...")
base_model.config.use_cache = False  # Required when using gradient checkpointing
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

# 3. LoRA configuration
lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["lora_target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
)

# 4. Apply LoRA adapters and ensure parameters are trainable
if "model" in globals() and isinstance(model, PeftModel):
    # Ensure LoRA adapter parameters have gradients enabled
    for name, param in model.named_parameters():
        if "lora_" in name:
            param.requires_grad = True
    trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if trainable_count > 0:
        print("ℹ️ Reusing active PEFT model with trainable adapters.")
    else:
        print("🔧 Re-applying LoRA adapters...")
        model = get_peft_model(base_model, lora_config)
else:
    print("🔧 Applying LoRA adapters...")
    model = get_peft_model(base_model, lora_config)

# 4b. Eliminate all BFloat16 tensors for Tesla T4 GPU compatibility:
# T4 does not support BFloat16 in PyTorch GradScaler.
# Cast non-quantized modules (embed_tokens, norm, lm_head), 4-bit compute_dtype, and adapters to FP16/FP32.
if not use_bf16:
    print("🔧 Enforcing FP16/FP32 precision across all layers for T4...")
    # A. Cast unquantized embedding and norm layers
    if hasattr(base_model, "model"):
        if hasattr(base_model.model, "embed_tokens") and base_model.model.embed_tokens is not None:
            base_model.model.embed_tokens.to(torch.float16)
        if hasattr(base_model.model, "norm") and base_model.model.norm is not None:
            base_model.model.norm.to(torch.float16)
    if hasattr(base_model, "lm_head") and base_model.lm_head is not None:
        base_model.lm_head.to(torch.float16)

    # B. Set compute_dtype to float16 on all 4-bit linear layers
    for module in base_model.modules():
        if isinstance(module, bnb.nn.Linear4bit):
            module.compute_dtype = torch.float16

    # C. Cast all remaining bfloat16 parameters and buffers
    conv_params = 0
    for name, param in model.named_parameters():
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float32 if param.requires_grad else torch.float16)
            conv_params += 1
    for name, buf in model.named_buffers():
        if buf.dtype == torch.bfloat16:
            buf.data = buf.data.to(torch.float16)
            conv_params += 1
    print(f"✅ Precision aligned: {conv_params} BFloat16 tensors converted to FP16/FP32.")

model.train()
model.print_trainable_parameters()

# 5. Training configuration (SFTConfig / TrainingArguments)
warmup_steps = max(1, int(CONFIG["max_steps"] * CONFIG.get("warmup_ratio", 0.03)))

# Select optimizer safely:
# Standard PyTorch AdamW uses only ~35MB VRAM for 4.3M LoRA params and avoids Python 3.13 bitsandbytes issues.
if is_bitsandbytes_available():
    chosen_optim = "paged_adamw_8bit"
else:
    chosen_optim = "adamw_torch"

sft_kwargs = dict(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    max_steps=CONFIG["max_steps"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    gradient_checkpointing=True,
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=warmup_steps,
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    eval_steps=CONFIG["eval_steps"],
    save_total_limit=CONFIG["save_total_limit"],
    fp16=not use_bf16,
    bf16=use_bf16,
    optim=chosen_optim,
    lr_scheduler_type="cosine",
    report_to="none",
    max_seq_length=CONFIG["max_length"],
    packing=False,
    dataset_text_field="text",
)

# Robust fallback across different transformers and trl versions
try:
    training_args = SFTConfig(**sft_kwargs, eval_strategy="steps" if val_split is not None else "no")
except TypeError:
    try:
        training_args = SFTConfig(**sft_kwargs, evaluation_strategy="steps" if val_split is not None else "no")
    except TypeError:
        from transformers import TrainingArguments
        ta_kwargs = {k: v for k, v in sft_kwargs.items() if k not in ("max_seq_length", "packing", "dataset_text_field")}
        training_args = TrainingArguments(**ta_kwargs, eval_strategy="steps" if val_split is not None else "no")

# 6. Initialize SFTTrainer
try:
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_split,
        eval_dataset=val_split,
        processing_class=tokenizer,
    )
except TypeError:
    try:
        trainer = SFTTrainer(
            model=model,
            args=training_args,
            train_dataset=train_split,
            eval_dataset=val_split,
            tokenizer=tokenizer,
        )
    except TypeError:
        trainer = SFTTrainer(
            model=model,
            args=training_args,
            train_dataset=train_split,
            eval_dataset=val_split,
            tokenizer=tokenizer,
            dataset_text_field="text",
            max_seq_length=CONFIG["max_length"],
            packing=False,
        )

print("
" + "="*60)
print("TRAINING CONFIGURATION")
print("="*60)
print(f"Effective batch size: {effective_batch_size}")
print(f"Max training steps:   {CONFIG['max_steps']}")
print(f"Warmup steps:         {warmup_steps}")
print(f"Optimizer:            {chosen_optim}")
print(f"Learning rate:        {CONFIG['learning_rate']}")
print(f"LoRA rank (r):        {CONFIG['lora_r']} | Alpha: {CONFIG['lora_alpha']}")
print(f"Precision:            {'BF16' if use_bf16 else 'FP16'}")
print("="*60)

# 7. Start Training
print("
🚀 Starting training...
")
train_result = trainer.train()

# 8. Save fine-tuned adapter weights and tokenizer
adapter_output = CONFIG.get("output_dir", "outputs/adapter")
trainer.save_model(adapter_output)
tokenizer.save_pretrained(adapter_output)
print(f"
💾 LoRA adapter and tokenizer saved successfully to: '{adapter_output}'")

# Re-enable cache for fast post-training inference
base_model.config.use_cache = True

# 9. Training Summary & Metrics
print("
" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
train_runtime = train_result.metrics.get("train_runtime", 1)
print(f"Total training time:  {train_runtime:.2f}s ({train_runtime / 60:.1f} min)")
print(f"Samples per second:   {train_result.metrics.get('train_samples_per_second', 0):.2f}")
print(f"Steps per second:     {train_result.metrics.get('train_steps_per_second', 0):.4f}")

total_tokens = CONFIG["max_steps"] * effective_batch_size * CONFIG["max_length"]
tokens_per_sec = total_tokens / train_runtime if train_runtime > 0 else 0
print(f"Approximate speed:    {tokens_per_sec:.1f} tokens/second")
print("="*60)


In [ ]:
# ----------------------------------------------------------------------------
# Post-Fine-Tuning Evaluation & Inference
# ----------------------------------------------------------------------------
import time
import torch

print("=" * 60)
print("POST-FINETUNING EVALUATION (WITH TRAINED LORA ADAPTER)")
print("=" * 60)

model.eval()
base_model.config.use_cache = True

# Generation configuration
use_sampling = CONFIG.get("do_sample", False) and (CONFIG.get("temperature", 0.0) > 0)

for i, test in enumerate(test_prompts, 1):
    # 1. Format as ChatML with assistant generation prompt
    formatted_prompt = chatml_format(test["prompt"])
    if not formatted_prompt.endswith("<|im_start|>assistant\n"):
        formatted_prompt += "<|im_start|>assistant\n"

    # 2. Tokenize and move to device
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(base_model.device)
    input_len = inputs["input_ids"].shape[1]

    # 3. Generate with adapter active
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CONFIG.get("max_new_tokens", 128),
            temperature=CONFIG.get("temperature", 0.0) if use_sampling else None,
            do_sample=use_sampling,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    elapsed = time.time() - start_time

    # 4. Extract generated tokens
    generated_tokens = outputs[0][input_len:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    tokens_generated = len(generated_tokens)
    tokens_per_sec = tokens_generated / elapsed if elapsed > 0 else 0

    print(f"\n[{i}] {test['title']}")
    print("-" * 60)
    print(f"Prompt: {test['prompt'][:100]}...")
    print(f"\nFine-Tuned Response:\n{response}")
    print(f"\nLatency: {elapsed:.2f}s | Tokens: {tokens_generated} | Speed: {tokens_per_sec:.1f} tok/s")
    print("-" * 60)

print("=" * 60)
